# Investment Research Multi-Agent System

This notebook contains the complete project implementation in one file and is organized as a single end-to-end workflow.

**Data ingestion → preprocessing → evidence assembly → news processing → planner/router → specialist agents → evaluator-optimizer → memory → final report**

## 1. Setup and Configuration

In [1]:
from __future__ import annotations

import ast
import datetime
import hashlib
import json
import math
import os
import random
import re
import time
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import requests
import yfinance as yf
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

# Notebook-safe project root.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env", override=True)

SEED = 42

PATHS = {
    "root": PROJECT_ROOT,
    "raw": PROJECT_ROOT / "data" / "raw",
    "processed": PROJECT_ROOT / "data" / "processed",
    "cache": PROJECT_ROOT / "data" / "cache",
    "docs": PROJECT_ROOT / "docs",
}

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)

def ensure_directories() -> None:
    for key, path in PATHS.items():
        if key != "root":
            path.mkdir(parents=True, exist_ok=True)

def get_target_tickers() -> list[str]:
    raw = os.getenv("TARGET_TICKERS", "AAPL")
    return [ticker.strip().upper() for ticker in raw.split(",") if ticker.strip()]

set_seed()
ensure_directories()

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)

TICKERS = get_target_tickers()

ALPHA_VANTAGE_API_KEY = os.getenv("ALPHA_VANTAGE_API_KEY", "")
MARKETAUX_API_KEY = os.getenv("MARKETAUX_API_KEY", "")
SEC_USER_AGENT = os.getenv("SEC_USER_AGENT", "")
LLM_API_KEY = os.getenv("LLM_API_KEY", "")
LLM_MODEL = os.getenv("LLM_MODEL", "gpt-4o-mini")
LLM_BASE_URL = os.getenv("LLM_BASE_URL", "").strip()

SEC_FORM4_LIMIT = int(os.getenv("SEC_FORM4_LIMIT", "20"))
NEWS_LIMIT_PER_TICKER = int(os.getenv("NEWS_LIMIT_PER_TICKER", "9"))
MARKETAUX_PAGE_SIZE = int(os.getenv("MARKETAUX_PAGE_SIZE", "3"))
NEWS_CHAIN_LIMIT = int(os.getenv("NEWS_CHAIN_LIMIT", "10"))

# Set to 0 to disable freshness filtering.
NEWS_MAX_AGE_DAYS = int(os.getenv("NEWS_MAX_AGE_DAYS", "365"))

FORCE_REFRESH = os.getenv("FORCE_REFRESH", "false").strip().lower() in {
    "1", "true", "yes"
}
FORCE_NEWS_CHAIN = os.getenv("FORCE_NEWS_CHAIN", "false").strip().lower() in {
    "1", "true", "yes"
}

print("Project root:", PROJECT_ROOT)
print("Target tickers:", TICKERS)
print("LLM model:", LLM_MODEL)
print("Force refresh:", FORCE_REFRESH)
print("Force news chain:", FORCE_NEWS_CHAIN)

print("Alpha Vantage key loaded:", bool(ALPHA_VANTAGE_API_KEY))


Project root: C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System
Target tickers: ['AAPL']
LLM model: openai/gpt-5-nano
Force refresh: False
Force news chain: False
Alpha Vantage key loaded: True


## 2. Data Ingestion

These functions were originally stored in the project source modules. They are included here directly so the notebook is standalone.

### 2.1 SEC EDGAR Form 4

In [2]:
TICKER_MAP_URL = "https://www.sec.gov/files/company_tickers.json"
SUBMISSIONS_URL = "https://data.sec.gov/submissions/CIK{cik}.json"
ARCHIVES_BASE_URL = "https://www.sec.gov/Archives/edgar/data"

OUTPUT_COLUMNS = [
    "ticker",
    "company_name",
    "owner_name",
    "owner_title",
    "transaction_date",
    "filing_date",
    "transaction_code",
    "acquired_disposed",
    "shares",
    "price",
    "accession_number",
    "source_url",
]


def _session(user_agent: str) -> requests.Session:
    """Create an SEC-friendly HTTP session."""
    if not user_agent or "@" not in user_agent:
        raise ValueError(
            "SEC_USER_AGENT must identify the application and include a contact email."
        )

    session = requests.Session()
    session.headers.update(
        {
            "User-Agent": user_agent,
            "Accept-Encoding": "gzip, deflate",
            "Host": "www.sec.gov",
        }
    )
    return session


def _get_json(
    session: requests.Session,
    url: str,
    sleep_seconds: float,
) -> dict[str, Any]:
    """GET a JSON resource while respecting a small delay between SEC requests."""
    response = session.get(url, timeout=30)
    response.raise_for_status()
    time.sleep(sleep_seconds)
    return response.json()


def _get_text(
    session: requests.Session,
    url: str,
    sleep_seconds: float,
) -> str:
    """GET a text/XML resource while respecting a small SEC request delay."""
    response = session.get(url, timeout=30)
    response.raise_for_status()
    time.sleep(sleep_seconds)
    return response.text


def get_ticker_cik_map(
    user_agent: str,
    sleep_seconds: float = 0.12,
) -> dict[str, str]:
    """Return a mapping like {'AAPL': '0000320193'}."""
    session = _session(user_agent)

    # company_tickers.json is served from www.sec.gov.
    payload = _get_json(session, TICKER_MAP_URL, sleep_seconds)

    return {
        item["ticker"].upper(): str(item["cik_str"]).zfill(10)
        for item in payload.values()
    }


def _issuer_submissions(
    cik: str,
    user_agent: str,
    sleep_seconds: float,
) -> dict[str, Any]:
    """Load the SEC submissions JSON for an issuer."""
    session = _session(user_agent)

    # data.sec.gov uses the same SEC User-Agent policy but a different host.
    session.headers.pop("Host", None)

    url = SUBMISSIONS_URL.format(cik=cik)
    return _get_json(session, url, sleep_seconds)


def _recent_form4_filings(
    submissions: dict[str, Any],
    limit: int,
) -> list[dict[str, str]]:
    """Extract recent Form 4 / Form 4-A metadata from submissions JSON."""
    recent = submissions.get("filings", {}).get("recent", {})

    forms = recent.get("form", [])
    accessions = recent.get("accessionNumber", [])
    filing_dates = recent.get("filingDate", [])

    filings: list[dict[str, str]] = []

    for form, accession, filing_date in zip(
        forms,
        accessions,
        filing_dates,
        strict=False,
    ):
        if form not in {"4", "4/A"}:
            continue

        filings.append(
            {
                "form": form,
                "accession_number": accession,
                "filing_date": filing_date,
            }
        )

        if len(filings) >= limit:
            break

    return filings


def _filing_directory_url(cik: str, accession_number: str) -> str:
    """Return the SEC Archives directory URL for a filing."""
    cik_no_leading_zeros = str(int(cik))
    accession_no_dashes = accession_number.replace("-", "")

    return (
        f"{ARCHIVES_BASE_URL}/"
        f"{cik_no_leading_zeros}/"
        f"{accession_no_dashes}"
    )


def _find_ownership_xml_url(
    session: requests.Session,
    cik: str,
    accession_number: str,
    sleep_seconds: float,
) -> str:
    """Find the raw ownership XML file from a filing's index.json."""
    directory_url = _filing_directory_url(cik, accession_number)
    index_url = f"{directory_url}/index.json"

    payload = _get_json(session, index_url, sleep_seconds)
    items = payload.get("directory", {}).get("item", [])

    names = [
        item.get("name", "")
        for item in items
        if item.get("name")
    ]

    # Most ownership filings use form4.xml. Prefer it explicitly.
    preferred_names = [
        name
        for name in names
        if name.lower() == "form4.xml"
    ]

    if preferred_names:
        return f"{directory_url}/{preferred_names[0]}"

    # Fallback: use an XML file that is not an index/XSD file.
    xml_candidates = [
        name
        for name in names
        if name.lower().endswith(".xml")
        and "index" not in name.lower()
        and not name.lower().endswith(".xsd")
    ]

    if not xml_candidates:
        raise ValueError(
            f"No ownership XML found for accession {accession_number}. "
            f"Files in filing directory: {names}"
        )

    return f"{directory_url}/{xml_candidates[0]}"


def _strip_namespaces(root: ET.Element) -> None:
    """Remove XML namespaces so tag lookup is consistent."""
    for element in root.iter():
        if "}" in element.tag:
            element.tag = element.tag.split("}", 1)[1]


def _text(
    parent: ET.Element | None,
    path: str,
) -> str | None:
    """Safely return stripped text from a nested XML path."""
    if parent is None:
        return None

    element = parent.find(path)
    if element is None or element.text is None:
        return None

    value = element.text.strip()
    return value if value else None


def _number(
    parent: ET.Element | None,
    path: str,
) -> float | None:
    """Safely parse a numeric value from XML."""
    value = _text(parent, path)

    if value is None:
        return None

    try:
        return float(value.replace(",", ""))
    except ValueError:
        return None


def _owner_title(owner: ET.Element) -> str | None:
    """Create a readable owner role/title from Form 4 relationship fields."""
    relationship = owner.find("reportingOwnerRelationship")
    if relationship is None:
        return None

    officer_title = _text(relationship, "officerTitle")
    if officer_title:
        return officer_title

    roles: list[str] = []

    role_fields = [
        ("isDirector", "Director"),
        ("isOfficer", "Officer"),
        ("isTenPercentOwner", "10% Owner"),
        ("isOther", "Other"),
    ]

    for field, label in role_fields:
        value = _text(relationship, field)
        if value in {"1", "true", "TRUE"}:
            roles.append(label)

    return ", ".join(roles) if roles else None


def _transaction_rows(
    root: ET.Element,
    ticker: str,
    company_name: str | None,
    filing_date: str,
    accession_number: str,
    source_url: str,
) -> list[dict[str, Any]]:
    """Flatten ownership XML transactions into row dictionaries."""
    owners = root.findall("reportingOwner")

    owner_records: list[tuple[str | None, str | None]] = []
    for owner in owners:
        owner_name = _text(
            owner,
            "reportingOwnerId/rptOwnerName",
        )
        owner_records.append(
            (
                owner_name,
                _owner_title(owner),
            )
        )

    if not owner_records:
        owner_records = [(None, None)]

    transactions: list[ET.Element] = []
    transactions.extend(
        root.findall(
            "nonDerivativeTable/nonDerivativeTransaction"
        )
    )
    transactions.extend(
        root.findall(
            "derivativeTable/derivativeTransaction"
        )
    )

    rows: list[dict[str, Any]] = []

    for transaction in transactions:
        transaction_date = _text(
            transaction,
            "transactionDate/value",
        )
        transaction_code = _text(
            transaction,
            "transactionCoding/transactionCode",
        )
        shares = _number(
            transaction,
            "transactionAmounts/transactionShares/value",
        )
        price = _number(
            transaction,
            "transactionAmounts/transactionPricePerShare/value",
        )
        acquired_disposed = _text(
            transaction,
            "transactionAmounts/transactionAcquiredDisposedCode/value",
        )

        for owner_name, owner_title in owner_records:
            rows.append(
                {
                    "ticker": ticker,
                    "company_name": company_name,
                    "owner_name": owner_name,
                    "owner_title": owner_title,
                    "transaction_date": transaction_date,
                    "filing_date": filing_date,
                    "transaction_code": transaction_code,
                    "acquired_disposed": acquired_disposed,
                    "shares": shares,
                    "price": price,
                    "accession_number": accession_number,
                    "source_url": source_url,
                }
            )

    return rows


def _parse_ownership_xml(
    xml_text: str,
    ticker: str,
    fallback_company_name: str | None,
    filing_date: str,
    accession_number: str,
    source_url: str,
) -> list[dict[str, Any]]:
    """Parse one raw Form 4 ownership XML document."""
    root = ET.fromstring(xml_text)
    _strip_namespaces(root)

    if root.tag != "ownershipDocument":
        raise ValueError(
            f"Expected ownershipDocument root, received {root.tag!r}"
        )

    company_name = (
        _text(root, "issuer/issuerName")
        or fallback_company_name
    )

    return _transaction_rows(
        root=root,
        ticker=ticker,
        company_name=company_name,
        filing_date=filing_date,
        accession_number=accession_number,
        source_url=source_url,
    )


def fetch_form4_transactions(
    tickers: list[str],
    user_agent: str,
    filings_per_ticker: int = 20,
    sleep_seconds: float = 0.12,
) -> pd.DataFrame:
    """Fetch recent SEC Form 4 transactions for one or more tickers."""
    ticker_map = get_ticker_cik_map(
        user_agent=user_agent,
        sleep_seconds=sleep_seconds,
    )

    archive_session = _session(user_agent)
    all_rows: list[dict[str, Any]] = []

    for raw_ticker in tickers:
        ticker = raw_ticker.strip().upper()

        cik = ticker_map.get(ticker)
        if cik is None:
            print(f"SEC: ticker not found in SEC ticker map: {ticker}")
            continue

        submissions = _issuer_submissions(
            cik=cik,
            user_agent=user_agent,
            sleep_seconds=sleep_seconds,
        )

        company_name = submissions.get("name")

        filings = _recent_form4_filings(
            submissions=submissions,
            limit=filings_per_ticker,
        )

        print(
            f"SEC: {ticker} | CIK {cik} | "
            f"{len(filings)} recent Form 4 filing(s)"
        )

        for filing in filings:
            accession_number = filing["accession_number"]

            try:
                xml_url = _find_ownership_xml_url(
                    session=archive_session,
                    cik=cik,
                    accession_number=accession_number,
                    sleep_seconds=sleep_seconds,
                )

                xml_text = _get_text(
                    archive_session,
                    xml_url,
                    sleep_seconds,
                )

                rows = _parse_ownership_xml(
                    xml_text=xml_text,
                    ticker=ticker,
                    fallback_company_name=company_name,
                    filing_date=filing["filing_date"],
                    accession_number=accession_number,
                    source_url=xml_url,
                )

                all_rows.extend(rows)

                print(
                    f"SEC: parsed {ticker} {accession_number} "
                    f"({len(rows)} transaction row(s))"
                )

            except (
                requests.RequestException,
                ET.ParseError,
                ValueError,
            ) as exc:
                print(
                    f"SEC: failed {ticker} {accession_number}: {exc}"
                )

    if not all_rows:
        return pd.DataFrame(columns=OUTPUT_COLUMNS)

    result = pd.DataFrame(all_rows)

    for column in OUTPUT_COLUMNS:
        if column not in result.columns:
            result[column] = pd.NA

    return result[OUTPUT_COLUMNS].reset_index(drop=True)


### 2.2 Alpha Vantage with yfinance Fallback

In [3]:
BASE_URL = "https://www.alphavantage.co/query"


def _alpha_vantage_request(params: dict[str, Any], api_key: str) -> dict[str, Any]:
    response = requests.get(
        BASE_URL,
        params={**params, "apikey": api_key},
        timeout=30,
    )
    response.raise_for_status()
    payload = response.json()

    for key in ("Error Message", "Information", "Note"):
        if key in payload:
            raise RuntimeError(payload[key])

    return payload


def fetch_daily_market_data(
    tickers: list[str],
    api_key: str,
) -> pd.DataFrame:
    """Fetch compact daily OHLCV data."""
    rows: list[dict[str, Any]] = []

    for ticker in tickers:
        symbol = ticker.strip().upper()

        payload = _alpha_vantage_request(
            {
                "function": "TIME_SERIES_DAILY",
                "symbol": symbol,
                "outputsize": "compact",
            },
            api_key,
        )

        for date, values in payload.get("Time Series (Daily)", {}).items():
            rows.append(
                {
                    "ticker": symbol,
                    "date": date,
                    "open": values.get("1. open"),
                    "high": values.get("2. high"),
                    "low": values.get("3. low"),
                    "close": values.get("4. close"),
                    "volume": values.get("5. volume"),
                }
            )

    return pd.DataFrame(rows)


In [4]:
def fetch_yfinance_fallback(tickers):
    """Fetch daily market data from yfinance and align its schema with Alpha Vantage."""

    all_data = []

    for ticker in tickers:
        print(f"Fetching {ticker} via yfinance fallback...")

        try:
            stock = yf.Ticker(ticker)
            df = stock.history(period="1mo").reset_index()

            if not df.empty:
                df["ticker"] = ticker.upper()
                df["date"] = df["Date"].dt.strftime("%Y-%m-%d")

                df = df.rename(
                    columns={
                        "Open": "open",
                        "High": "high",
                        "Low": "low",
                        "Close": "close",
                        "Volume": "volume",
                    }
                )

                cols = [
                    "date",
                    "ticker",
                    "open",
                    "high",
                    "low",
                    "close",
                    "volume",
                ]
                all_data.append(df[cols])

        except Exception as exc:
            print(f"Failed to fetch {ticker} from yfinance: {exc}")

    if all_data:
        return pd.concat(all_data, ignore_index=True)

    return pd.DataFrame()

### 2.3 Marketaux Financial News

In [5]:
"""Marketaux financial-news ingestion helpers."""


BASE_URL = "https://api.marketaux.com/v1/news/all"


def _marketaux_request(params: dict[str, Any], api_token: str) -> dict[str, Any]:
    response = requests.get(
        BASE_URL,
        params={**params, "api_token": api_token},
        timeout=30,
    )
    response.raise_for_status()
    payload = response.json()

    meta = payload.get("meta", {})
    if meta.get("code"):
        raise RuntimeError(
            meta.get("message")
            or f"Marketaux error: {meta.get('code')}"
        )

    return payload


def fetch_financial_news(
    tickers: list[str],
    api_token: str,
    limit_per_ticker: int = 9,
    page_size: int = 3,
    pause_seconds: float = 0.5,
) -> pd.DataFrame:
    """Fetch recent financial news with pagination."""
    rows: list[dict[str, Any]] = []

    for ticker in tickers:
        symbol = ticker.strip().upper()
        ticker_rows: list[dict[str, Any]] = []
        pages_needed = math.ceil(limit_per_ticker / page_size)

        for page in range(1, pages_needed + 1):
            remaining = limit_per_ticker - len(ticker_rows)
            if remaining <= 0:
                break

            payload = _marketaux_request(
                {
                    "symbols": symbol,
                    "filter_entities": "true",
                    "language": "en",
                    "limit": min(page_size, remaining),
                    "page": page,
                },
                api_token,
            )

            articles = payload.get("data", [])
            if not articles:
                break

            for article in articles:
                ticker_rows.append(
                    {
                        "ticker": symbol,
                        "published_at": article.get("published_at"),
                        "title": article.get("title"),
                        "description": article.get("description"),
                        "content": (
                            article.get("snippet")
                            or article.get("description")
                        ),
                        "source": article.get("source"),
                        "url": article.get("url"),
                    }
                )

                if len(ticker_rows) >= limit_per_ticker:
                    break

            if page < pages_needed:
                time.sleep(pause_seconds)

        rows.extend(ticker_rows[:limit_per_ticker])

    return pd.DataFrame(rows)


### 2.4 Run Data Ingestion

In [6]:
RAW_OUTPUTS = {
    "sec": PATHS["raw"] / "sec_form4.csv",
    "market": PATHS["raw"] / "market_data.csv",
    "news": PATHS["raw"] / "news.csv",
}

# SEC
sec_path = RAW_OUTPUTS["sec"]
if sec_path.exists() and not FORCE_REFRESH:
    print("Using existing SEC snapshot:", sec_path)
    sec_raw = pd.read_csv(sec_path)
else:
    if not SEC_USER_AGENT or "@" not in SEC_USER_AGENT:
        raise ValueError("SEC_USER_AGENT is missing or invalid.")
    sec_raw = fetch_form4_transactions(
        tickers=TICKERS,
        user_agent=SEC_USER_AGENT,
        filings_per_ticker=SEC_FORM4_LIMIT,
    )
    sec_raw.to_csv(sec_path, index=False)

# Market
market_path = RAW_OUTPUTS["market"]

if market_path.exists() and not FORCE_REFRESH:
    print("Using existing market snapshot:", market_path)
    market_raw = pd.read_csv(market_path)
else:
    print("Fetching Alpha Vantage market data...")

    try:
        market_raw = fetch_daily_market_data(
            tickers=TICKERS,
            api_key=ALPHA_VANTAGE_API_KEY,
        )

        if market_raw is None or market_raw.empty:
            raise RuntimeError(
                "Alpha Vantage returned empty data or hit its rate limit."
            )

        print("Successfully fetched market data from Alpha Vantage.")

    except Exception as exc:
        print(f"Alpha Vantage fetch failed: {exc}")
        print("Switching to yfinance fallback...")
        market_raw = fetch_yfinance_fallback(TICKERS)

    if not market_raw.empty:
        market_raw.to_csv(market_path, index=False)
        print("Saved market snapshot:", market_path)
    else:
        print("Warning: failed to retrieve market data from both sources.")

# News
news_path = RAW_OUTPUTS["news"]
if news_path.exists() and not FORCE_REFRESH:
    print("Using existing news snapshot:", news_path)
    news_raw = pd.read_csv(news_path)
else:
    if not MARKETAUX_API_KEY:
        raise ValueError("MARKETAUX_API_KEY is missing from .env.")
    news_raw = fetch_financial_news(
        tickers=TICKERS,
        api_token=MARKETAUX_API_KEY,
        limit_per_ticker=NEWS_LIMIT_PER_TICKER,
        page_size=MARKETAUX_PAGE_SIZE,
    )
    news_raw.to_csv(news_path, index=False)

display(pd.DataFrame([
    {"source": "SEC Form 4", "rows": len(sec_raw), "path": str(sec_path)},
    {"source": "Market", "rows": len(market_raw), "path": str(market_path)},
    {"source": "News", "rows": len(news_raw), "path": str(news_path)},
]))


Using existing SEC snapshot: C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System\data\raw\sec_form4.csv
Using existing market snapshot: C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System\data\raw\market_data.csv
Using existing news snapshot: C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System\data\raw\news.csv


,source,rows,path
0,SEC Form 4,61,C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System\data\raw\sec_form4.csv
1,Market,100,C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System\data\raw\market_data.csv
2,News,50,C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System\data\raw\news.csv


## 3. Data Preprocessing

This section cleans the three raw data sources and implements the stricter financial-news relevance rule requested during final team review.

In [7]:
def normalize_column_names(df):
    df = df.copy()
    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    return df

def normalize_ticker(series):
    return (
        series.astype("string")
        .str.strip()
        .str.upper()
        .replace({"": pd.NA, "NAN": pd.NA, "NONE": pd.NA})
    )

def clean_text(series):
    return (
        series.astype("string")
        .str.replace(r"<[^>]+>", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

def to_utc(series):
    return pd.to_datetime(series, errors="coerce", utc=True)

def report_rows(name, before, after):
    print(f"{name}: {before:,} -> {after:,} rows ({before-after:,} removed)")

def fingerprint(df):
    data = pd.util.hash_pandas_object(
        df.sort_index(axis=1).fillna("<NA>"),
        index=True,
    ).values.tobytes()
    return hashlib.sha256(data).hexdigest()[:16]

COMPANY_ALIASES = {
    "AAPL": ["apple", "aapl"],
    "MSFT": ["microsoft", "msft"],
    "NVDA": ["nvidia", "nvda"],
}

def _mention_count(text, aliases):
    text = "" if pd.isna(text) else str(text).lower()
    total = 0
    for alias in aliases:
        # Exact word-style matching avoids loose substring matches.
        pattern = rf"(?<!\w){re.escape(alias.lower())}(?!\w)"
        total += len(re.findall(pattern, text))
    return total

def get_news_relevance_details(row):
    ticker = str(row.get("ticker", "")).upper().strip()
    aliases = COMPANY_ALIASES.get(ticker, [ticker.lower()])

    title = row.get("title", "")
    body = " ".join([
        "" if pd.isna(row.get("description")) else str(row.get("description")),
        "" if pd.isna(row.get("content")) else str(row.get("content")),
    ])

    title_mentions = _mention_count(title, aliases)
    body_mentions = _mention_count(body, aliases)

    # Final-review fix:
    # title mention = strong relevance;
    # otherwise require repeated body mentions so a one-off reference does not pass.
    if title_mentions >= 1:
        return pd.Series([
            2,
            "target company/ticker appears in title",
            title_mentions,
            body_mentions,
        ])

    if body_mentions >= 2:
        return pd.Series([
            1,
            "target company/ticker appears multiple times in article body",
            title_mentions,
            body_mentions,
        ])

    return pd.Series([
        0,
        "target company/ticker is absent or only mentioned incidentally",
        title_mentions,
        body_mentions,
    ])


In [8]:
def preprocess_sec(df):
    before = len(df)
    df = normalize_column_names(df)

    required = ["ticker", "transaction_date", "owner_name"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"SEC data missing required columns: {missing}")

    optional = [
        "company_name", "owner_title", "filing_date",
        "transaction_code", "acquired_disposed",
        "shares", "price", "accession_number", "source_url",
    ]
    for col in optional:
        if col not in df.columns:
            df[col] = pd.NA

    df["ticker"] = normalize_ticker(df["ticker"])
    for col in ["owner_name", "company_name", "owner_title"]:
        df[col] = clean_text(df[col])

    df["transaction_date"] = to_utc(df["transaction_date"])
    df["filing_date"] = to_utc(df["filing_date"])
    df["shares"] = pd.to_numeric(df["shares"], errors="coerce")
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df["transaction_value"] = df["shares"] * df["price"]
    df["transaction_code"] = df["transaction_code"].astype("string").str.strip().str.upper()
    df["acquired_disposed"] = df["acquired_disposed"].astype("string").str.strip().str.upper()

    df = df.dropna(subset=["ticker", "transaction_date", "owner_name"])
    df = df.drop_duplicates(
        subset=[
            "accession_number", "ticker", "owner_name", "transaction_date",
            "transaction_code", "shares", "price",
        ]
    )
    df = df.sort_values(["ticker", "transaction_date"]).reset_index(drop=True)

    report_rows("SEC Form 4", before, len(df))
    return df


def preprocess_market(df):
    before = len(df)
    df = normalize_column_names(df)

    required = ["ticker", "date", "close"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Market data missing required columns: {missing}")

    for col in ["open", "high", "low", "adj_close", "volume"]:
        if col not in df.columns:
            df[col] = pd.NA

    df["ticker"] = normalize_ticker(df["ticker"])
    df["date"] = to_utc(df["date"])

    for col in ["open", "high", "low", "close", "adj_close", "volume"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["ticker", "date", "close"])
    df = df.drop_duplicates(subset=["ticker", "date"])
    df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

    df["daily_return"] = df.groupby("ticker")["close"].pct_change()
    df["volume_change"] = df.groupby("ticker")["volume"].pct_change()

    if "adj_close" in df.columns and df["adj_close"].isna().all():
        df = df.drop(columns=["adj_close"])

    report_rows("Market", before, len(df))
    return df


def preprocess_news(df):
    before = len(df)
    df = normalize_column_names(df)

    required = ["published_at", "title"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"News data missing required columns: {missing}")

    optional = [
        "ticker", "company_name", "description",
        "content", "source", "url",
    ]
    for col in optional:
        if col not in df.columns:
            df[col] = pd.NA

    df["ticker"] = normalize_ticker(df["ticker"])
    df["published_at"] = to_utc(df["published_at"])

    for col in ["title", "description", "content", "company_name", "source"]:
        df[col] = clean_text(df[col])

    with_url = df[df["url"].notna()].drop_duplicates(subset=["url"])
    without_url = df[df["url"].isna()].drop_duplicates(
        subset=["title", "published_at"]
    )
    df = pd.concat([with_url, without_url], ignore_index=True)

    df = df.dropna(subset=["ticker", "published_at", "title"])

    df["text"] = (
        df["title"].fillna("")
        + ". "
        + df["description"].fillna("")
        + " "
        + df["content"].fillna("")
    )
    df["text"] = clean_text(df["text"])

    df[[
        "relevance_score",
        "relevance_reason",
        "title_mentions",
        "body_mentions",
    ]] = df.apply(get_news_relevance_details, axis=1)

    # Optional freshness check. Set NEWS_MAX_AGE_DAYS=0 to keep all dates.
    if NEWS_MAX_AGE_DAYS > 0:
        now_utc = pd.Timestamp.now(tz="UTC")
        df["article_age_days"] = (now_utc - df["published_at"]).dt.days
        df["is_fresh"] = df["article_age_days"].between(0, NEWS_MAX_AGE_DAYS)
    else:
        df["article_age_days"] = pd.NA
        df["is_fresh"] = True

    before_filter = len(df)
    df = df[
        (df["relevance_score"] >= 1)
        & (df["is_fresh"])
    ].copy()

    print(
        "News relevance/freshness filter:",
        f"{before_filter:,} -> {len(df):,} rows "
        f"({before_filter - len(df):,} removed)",
    )

    df = df.sort_values(
        ["relevance_score", "published_at"],
        ascending=[False, False],
    ).reset_index(drop=True)

    report_rows("News", before, len(df))
    return df


In [9]:
processed = {
    "sec": preprocess_sec(sec_raw),
    "market": preprocess_market(market_raw),
    "news": preprocess_news(news_raw),
}

OUTPUT_FILES = {
    "sec": PATHS["processed"] / "sec_form4_clean.csv",
    "market": PATHS["processed"] / "market_data_clean.csv",
    "news": PATHS["processed"] / "news_clean.csv",
}

for name, frame in processed.items():
    frame.to_csv(OUTPUT_FILES[name], index=False)
    print("Saved:", OUTPUT_FILES[name])

quality_df = pd.DataFrame([
    {
        "dataset": name,
        "rows": len(frame),
        "columns": len(frame.columns),
        "duplicate_rows": int(frame.duplicated().sum()),
        "missing_cells": int(frame.isna().sum().sum()),
        "fingerprint": fingerprint(frame),
    }
    for name, frame in processed.items()
])

display(quality_df)

display(
    processed["news"][
        [
            "ticker",
            "published_at",
            "title",
            "relevance_score",
            "relevance_reason",
            "title_mentions",
            "body_mentions",
            "article_age_days",
        ]
    ].head(20)
)

manifest = {
    "seed": SEED,
    "datasets": {
        name: {
            "rows": len(frame),
            "columns": list(frame.columns),
            "fingerprint": fingerprint(frame),
            "output_file": str(OUTPUT_FILES[name]),
        }
        for name, frame in processed.items()
    },
}

manifest_path = PATHS["processed"] / "preprocessing_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Saved:", manifest_path)


SEC Form 4: 61 -> 57 rows (4 removed)
Market: 100 -> 100 rows (0 removed)
News relevance/freshness filter: 48 -> 18 rows (30 removed)
News: 50 -> 18 rows (32 removed)
Saved: C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System\data\processed\sec_form4_clean.csv
Saved: C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System\data\processed\market_data_clean.csv
Saved: C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System\data\processed\news_clean.csv


,dataset,rows,columns,duplicate_rows,missing_cells,fingerprint
0,sec,57,13,0,48,2a9558af6146e008
1,market,100,9,0,2,4caa85a9db2f5407
2,news,18,15,0,18,2431305defb1671a


,ticker,published_at,title,relevance_score,relevance_reason,title_mentions,body_mentions,article_age_days
0,AAPL,2026-09-20 16:37:00+00:00,"Prediction: Even With the $1,999 Price Tag, Apple's Foldable iPhone Duo Will Be a Hit and Apple Will Join Nvidia in the $5 Trillion Club.",2,target company/ticker appears in title,2,3,14
1,AAPL,2026-09-20 16:25:40+00:00,AAPL Looks 17.3% Overvalued on GF Value™ as Insider Selling Persists,2,target company/ticker appears in title,1,4,14
2,AAPL,2026-09-20 11:30:29+00:00,Apple Enhances Durability with New A20 Pro Chip and Ceramic Shield in iPhone 18 Pro,2,target company/ticker appears in title,1,2,14
3,AAPL,2026-09-19 12:50:00+00:00,John Ternus's First iPhone Launch Prompted a Bank of America Price Target Cut. Is Apple Stock a Buy?,2,target company/ticker appears in title,1,3,15
4,AAPL,2026-09-18 18:11:54+00:00,Apple Raised Stakes with its $2000-Priced Foldable. But Can it Be the New Growth Engine?,2,target company/ticker appears in title,1,4,16
5,AAPL,2026-09-18 17:57:20+00:00,AAPL Looks 16.9% Overvalued on GF Value™ Amid Strong Upgrade Cycle Outlook,2,target company/ticker appears in title,1,4,16
6,AAPL,2026-09-18 16:55:26+00:00,META Looks 21.3% Undervalued on GF Value™ as Muse App Tops Apple Store,2,target company/ticker appears in title,1,2,16
7,AAPL,2026-09-18 15:30:05+00:00,Apple Stock Has a New Opportunity Investors Can’t Afford to Ignore,2,target company/ticker appears in title,1,2,16
8,AAPL,2026-09-18 13:30:58+00:00,AAPL Maintained by Evercore ISI Group -- Price Target Raised to $380,2,target company/ticker appears in title,1,8,16
9,AAPL,2026-09-18 13:30:00+00:00,"Apple Stock: The $1,999 iPhone Hides A Bigger Problem (NASDAQ:AAPL)",2,target company/ticker appears in title,2,5,16


Saved: C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System\data\processed\preprocessing_manifest.json


## 4. Evidence Assembly

In [10]:
def assemble_daily_evidence(
    market: pd.DataFrame,
    news: pd.DataFrame,
    sec: pd.DataFrame,
) -> pd.DataFrame:
    """Create one ticker/date evidence table without making investment judgments."""
    market = market.copy()
    news = news.copy()
    sec = sec.copy()

    market["date"] = pd.to_datetime(market["date"], utc=True).dt.normalize()
    news["date"] = pd.to_datetime(news["published_at"], utc=True).dt.normalize()
    sec["date"] = pd.to_datetime(sec["transaction_date"], utc=True).dt.normalize()

    news_daily = (
        news.groupby(["ticker", "date"], dropna=False)
        .agg(
            news_count=("title", "size"),
            news_titles=("title", lambda values: " || ".join(values.dropna().astype(str).head(10))),
            news_urls=("url", lambda values: " || ".join(values.dropna().astype(str).head(10))),
        )
        .reset_index()
    )

    sec_daily = (
        sec.groupby(["ticker", "date"], dropna=False)
        .agg(
            insider_transaction_count=("owner_name", "size"),
            insider_total_value=("transaction_value", "sum"),
            insider_names=(
                "owner_name",
                lambda values: " || ".join(
                    values.dropna().astype(str).unique()[:10]
                ),
            ),
        )
        .reset_index()
    )

    evidence = market.merge(news_daily, on=["ticker", "date"], how="outer")
    evidence = evidence.merge(sec_daily, on=["ticker", "date"], how="outer")
    evidence["news_count"] = evidence["news_count"].fillna(0).astype("int64")

    if "adj_close" in evidence and evidence["adj_close"].isna().all():
        evidence = evidence.drop(columns="adj_close")

    evidence = evidence.sort_values(["ticker", "date"]).reset_index(drop=True)
    return evidence


In [11]:
evidence = assemble_daily_evidence(
    market=processed["market"],
    news=processed["news"],
    sec=processed["sec"],
)

evidence_path = PATHS["processed"] / "daily_evidence.csv"
evidence.to_csv(evidence_path, index=False)

print("Saved:", evidence_path)
print("Evidence rows:", len(evidence))
display(evidence.head(10))


Saved: C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System\data\processed\daily_evidence.csv
Evidence rows: 108


,ticker,date,open,high,low,close,volume,daily_return,volume_change,news_count,news_titles,news_urls,insider_transaction_count,insider_total_value,insider_names
0,AAPL,2026-02-24 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,<NA>,NaN,1.0,0.00,WAGNER SUSAN
1,AAPL,2026-03-15 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,<NA>,NaN,2.0,8135903.36,Newstead Jennifer
2,AAPL,2026-04-01 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,<NA>,NaN,15.0,34320628.17,O'BRIEN DEIRDRE || Khan Sabih || COOK TIMOTHY D
3,AAPL,2026-04-02 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,<NA>,NaN,8.0,24173072.77,O'BRIEN DEIRDRE || COOK TIMOTHY D
4,AAPL,2026-04-15 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,<NA>,NaN,10.0,1514654.55,Borders Ben || Parekh Kevan
5,AAPL,2026-04-23 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,<NA>,NaN,1.0,421850.00,Parekh Kevan
6,AAPL,2026-04-28 00:00:00+00:00,272.335,273.23,268.66,270.71,40018940.0,NaN,NaN,0,<NA>,NaN,NaN,NaN,<NA>
7,AAPL,2026-04-29 00:00:00+00:00,267.550,271.04,267.04,270.17,30047869.0,-0.001995,-0.249159,0,<NA>,NaN,NaN,NaN,<NA>
8,AAPL,2026-04-30 00:00:00+00:00,270.500,276.00,268.14,271.35,91848230.0,0.004368,2.056730,0,<NA>,NaN,NaN,NaN,<NA>
9,AAPL,2026-05-01 00:00:00+00:00,278.855,287.22,278.37,280.14,79915442.0,0.032394,-0.129919,0,<NA>,NaN,NaN,NaN,<NA>


## 5. News Processing Chain

The chain performs **classify → extract → summarize**. The final version passes the publication date and source article into the prompts and checks any year used by the summary against the source evidence.

In [12]:
NEWS_INPUT = PATHS["processed"] / "news_clean.csv"
NEWS_OUTPUT = PATHS["processed"] / "news_research_results.csv"

NEWS_CATEGORIES = [
    "earnings",
    "product_service",
    "management",
    "regulation_legal",
    "merger_acquisition",
    "analyst_investor",
    "macro_market",
    "other",
]

def build_llm_client():
    if not LLM_API_KEY:
        raise ValueError("LLM_API_KEY is missing from .env.")
    if LLM_BASE_URL:
        return OpenAI(api_key=LLM_API_KEY, base_url=LLM_BASE_URL)
    return OpenAI(api_key=LLM_API_KEY)

news_client = build_llm_client()

def parse_json_response(raw_text):
    text = raw_text.strip()

    if text.startswith("```"):
        text = text.strip("`")
        if text.lower().startswith("json"):
            text = text[4:].strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, dict):
                return parsed
        except (ValueError, SyntaxError):
            pass

    raise ValueError(f"Model response was not valid JSON: {raw_text}")

def call_llm_json(system_prompt, user_prompt, temperature=0):
    response = news_client.chat.completions.create(
        model=LLM_MODEL,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    content = response.choices[0].message.content
    if not content:
        raise ValueError("The model returned an empty response.")
    return parse_json_response(content)


In [13]:
def classify_article(ticker, title, published_at, article_text):
    categories = ", ".join(NEWS_CATEGORIES)

    system_prompt = """
You classify financial news for an investment research pipeline.
Use only the supplied article.
Do not make investment recommendations.
Return valid JSON only.
""".strip()

    user_prompt = f"""
Target ticker: {ticker}
Publication date: {published_at}
Title: {title}

Classify this article into exactly one category:
{categories}

Return:
{{
  "category": "one_category_from_the_list",
  "classification_reason": "one concise sentence"
}}

Article:
{article_text}
""".strip()

    result = call_llm_json(system_prompt, user_prompt)
    if result.get("category") not in NEWS_CATEGORIES:
        raise ValueError(f"Invalid category returned: {result.get('category')}")
    return result


def extract_article_facts(ticker, title, published_at, article_text, category):
    system_prompt = """
You extract structured evidence from financial news.
Use only facts explicitly supported by the supplied article.
Do not infer missing facts.
Never invent or replace event dates.
Return valid JSON only.
""".strip()

    user_prompt = f"""
Target ticker: {ticker}
Publication date: {published_at}
Title: {title}
Category: {category}

Return:
{{
  "event": "one-sentence description of the primary event",
  "key_facts": ["fact 1", "fact 2"],
  "people": ["person name"],
  "organizations": ["organization name"],
  "financial_numbers": ["financial or quantitative value exactly as stated"]
}}

Use empty lists where needed.

Article:
{article_text}
""".strip()

    result = call_llm_json(system_prompt, user_prompt)

    for field in ["key_facts", "people", "organizations", "financial_numbers"]:
        value = result.get(field)
        if value is None:
            result[field] = []
        elif not isinstance(value, list):
            result[field] = [str(value)]

    result["event"] = result.get("event", "")
    return result


def summarize_article(
    ticker,
    title,
    published_at,
    article_text,
    category,
    extracted_facts,
):
    evidence_json = json.dumps(
        extracted_facts,
        ensure_ascii=False,
        indent=2,
    )

    system_prompt = """
You write concise evidence-grounded financial research summaries.

Use only the supplied source article and structured evidence.
Do not introduce new facts, dates, or years.
The publication date is metadata; do not present it as the event date unless the source supports that.
If the source refers to a historical event, make that relationship clear.
Do not make a buy, sell, or hold recommendation.
Return valid JSON only.
""".strip()

    user_prompt = f"""
Target ticker: {ticker}
Publication date: {published_at}
Title: {title}
Category: {category}

Structured evidence:
{evidence_json}

Source article:
{article_text}

Return exactly:
{{
  "summary": "2-4 sentence research summary"
}}
""".strip()

    return call_llm_json(system_prompt, user_prompt)


def years_in_text(text):
    return set(
        re.findall(
            r"\b(?:19|20)\d{2}\b",
            "" if pd.isna(text) else str(text),
        )
    )

def validate_summary_dates(summary, article_text, published_at):
    source_years = years_in_text(article_text)
    publication_year = str(pd.Timestamp(published_at).year)
    allowed_years = source_years | {publication_year}
    summary_years = years_in_text(summary)
    unsupported = sorted(summary_years - allowed_years)

    return {
        "date_grounded": len(unsupported) == 0,
        "source_years": sorted(source_years),
        "publication_year": publication_year,
        "summary_years": sorted(summary_years),
        "unsupported_years": unsupported,
    }


def run_news_chain(article):
    ticker = article["ticker"]
    title = article["title"]
    published_at = article["published_at"]
    article_text = article["text"]

    classification = classify_article(
        ticker,
        title,
        published_at,
        article_text,
    )

    extraction = extract_article_facts(
        ticker,
        title,
        published_at,
        article_text,
        classification["category"],
    )

    summary_result = summarize_article(
        ticker,
        title,
        published_at,
        article_text,
        classification["category"],
        extraction,
    )

    summary = summary_result.get("summary", "")
    date_check = validate_summary_dates(
        summary,
        article_text,
        published_at,
    )

    if not date_check["date_grounded"]:
        raise ValueError(
            "Summary introduced unsupported year(s): "
            + ", ".join(date_check["unsupported_years"])
        )

    return {
        "ticker": ticker,
        "published_at": published_at,
        "title": title,
        "source": article.get("source"),
        "url": article.get("url"),
        "relevance_score": article.get("relevance_score"),
        "relevance_reason": article.get("relevance_reason"),
        "category": classification["category"],
        "classification_reason": classification.get("classification_reason", ""),
        "event": extraction.get("event", ""),
        "key_facts": extraction.get("key_facts", []),
        "people": extraction.get("people", []),
        "organizations": extraction.get("organizations", []),
        "financial_numbers": extraction.get("financial_numbers", []),
        "summary": summary,
        "date_grounded": date_check["date_grounded"],
        "summary_years": date_check["summary_years"],
        "source_years": date_check["source_years"],
    }


In [14]:
news = processed["news"].copy()
news["published_at"] = pd.to_datetime(
    news["published_at"],
    errors="coerce",
    utc=True,
)
news["relevance_score"] = pd.to_numeric(
    news["relevance_score"],
    errors="coerce",
)

news = (
    news.dropna(subset=["ticker", "published_at", "title", "text"])
    .sort_values(
        ["relevance_score", "published_at"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

existing_results = pd.DataFrame()

if NEWS_OUTPUT.exists() and not FORCE_NEWS_CHAIN:
    existing_results = pd.read_csv(NEWS_OUTPUT)

completed_urls = set()
if not existing_results.empty and "url" in existing_results.columns:
    completed_urls = set(
        existing_results["url"].dropna().astype(str)
    )

articles_to_process = news.copy()

if completed_urls and not FORCE_NEWS_CHAIN:
    articles_to_process = articles_to_process[
        ~articles_to_process["url"].astype(str).isin(completed_urls)
    ]

articles_to_process = articles_to_process.head(NEWS_CHAIN_LIMIT)

new_results = []
failed_articles = []

for _, article in articles_to_process.iterrows():
    print("Processing:", article["ticker"], "-", article["title"][:90])

    try:
        new_results.append(run_news_chain(article))
    except Exception as exc:
        failed_articles.append({
            "ticker": article["ticker"],
            "title": article["title"],
            "error": f"{type(exc).__name__}: {exc}",
        })

new_results_df = pd.DataFrame(new_results)

if existing_results.empty:
    results = new_results_df.copy()
elif new_results_df.empty:
    results = existing_results.copy()
else:
    results = pd.concat(
        [existing_results, new_results_df],
        ignore_index=True,
    )

if not results.empty:
    results = results.drop_duplicates(
        subset=["url"],
        keep="last",
    )
    results.to_csv(NEWS_OUTPUT, index=False)

print("New successful rows:", len(new_results_df))
print("Failed rows:", len(failed_articles))
print("Total saved rows:", len(results))

if failed_articles:
    display(pd.DataFrame(failed_articles))

if not results.empty:
    display(
        results[
            [
                "ticker",
                "published_at",
                "title",
                "relevance_score",
                "category",
                "summary",
                "date_grounded",
            ]
        ].tail(20)
    )


Processing: AAPL - Apple's Quality Is Tempting, But I'm Put Off By The Valuation (NASDAQ:AAPL)
Processing: AAPL - AAPL Reiterated by B of A Securities -- Maintains Price Target at $370
Processing: AAPL - Apple (AAPL) Faces a $2.7 Billion UK Lawsuit over its App Tracking Rules
Processing: AAPL - Apple: Reality Check Coming, But I'm Not Selling (Downgrade) (NASDAQ:AAPL)
Processing: AAPL - OpenAI Fires Back at Apple (AAPL)
Processing: AAPL - AAPL Looks 16.2% Overvalued on GF Value™
Processing: AAPL - Mag 7 Laggard Meta Has Quietly Turned Positive for 2026. What Comes Next.
Processing: AAPL - The Same Stock but in 2 Different IRAs: By Retirement, the Tax Bill Is the Only Thing That
New successful rows: 8
Failed rows: 0
Total saved rows: 18


,ticker,published_at,title,relevance_score,category,summary,date_grounded
0,AAPL,2026-09-20 16:37:00+00:00,"Prediction: Even With the $1,999 Price Tag, Apple's Foldable iPhone Duo Will Be a Hit and Apple Will Join Nvidia in the $5 Trillion Club.",2,product_service,"The main event is a prediction that Apple's foldable iPhone Duo, priced at $1,999, will be a hit and will see Apple join NVIDIA in the $...",NaN
1,AAPL,2026-09-20 16:25:40+00:00,AAPL Looks 17.3% Overvalued on GF Value™ as Insider Selling Persists,2,analyst_investor,Apple Inc. is reportedly 17.3% overvalued on GF Value™ as insider selling persists. The key facts are that AAPL is shown as 17.3% overva...,NaN
2,AAPL,2026-09-20 11:30:29+00:00,Apple Enhances Durability with New A20 Pro Chip and Ceramic Shield in iPhone 18 Pro,2,product_service,"Apple unveiled the A20 Pro chip, Ceramic Shield in the iPhone 18 Pro, and an innovative folding screen on September 20, 2026. The key fa...",NaN
3,AAPL,2026-09-19 12:50:00+00:00,John Ternus's First iPhone Launch Prompted a Bank of America Price Target Cut. Is Apple Stock a Buy?,2,analyst_investor,"Bank of America cut Apple's price target after John Ternus's first iPhone launch event as Apple's CEO, where Apple unveiled the iPhone 1...",NaN
4,AAPL,2026-09-18 18:11:54+00:00,Apple Raised Stakes with its $2000-Priced Foldable. But Can it Be the New Growth Engine?,2,product_service,"Apple released its first foldable phone, the Duo, priced at $2000. The Duo is Apple's first foldable phone and represents the most signi...",NaN
5,AAPL,2026-09-18 17:57:20+00:00,AAPL Looks 16.9% Overvalued on GF Value™ Amid Strong Upgrade Cycle Outlook,2,analyst_investor,Apple is viewed as 16.9% overvalued on GF Value™ amid a strong upgrade cycle outlook. GF Value™ indicates Apple (AAPL) is 16.9% overvalu...,NaN
6,AAPL,2026-09-18 16:55:26+00:00,META Looks 21.3% Undervalued on GF Value™ as Muse App Tops Apple Store,2,product_service,"Meta Platforms Inc.'s Muse app topped Apple's U.S. App Store, reaching No. 1 on September 18, 2026. Key facts show Muse surged to No. 1 ...",NaN
7,AAPL,2026-09-18 15:30:05+00:00,Apple Stock Has a New Opportunity Investors Can’t Afford to Ignore,2,earnings,"Apple reported its ninth consecutive earnings beat, including a ninth consecutive EPS beat. Shares are up 23% this year, and the company...",NaN
8,AAPL,2026-09-18 13:30:58+00:00,AAPL Maintained by Evercore ISI Group -- Price Target Raised to $380,2,analyst_investor,"Evercore ISI Group maintained an Outperform rating for Apple Inc (AAPL) and raised its price target to $380 on September 18, 2026. This ...",NaN
9,AAPL,2026-09-18 13:30:00+00:00,"Apple Stock: The $1,999 iPhone Hides A Bigger Problem (NASDAQ:AAPL)",2,analyst_investor,"An analyst argues that Apple stock is a sell due to a bigger problem hinted by the $1,999 iPhone. The article cites the $1,999 iPhone, n...",NaN


### 5.1 News Quality Checks

These checks make the two teammate observations visible in the final demo instead of leaving them as informal notes.

In [15]:
if results.empty:
    print("No processed news results are available.")
else:
    qa_rows = [
        {
            "check": "All summaries present",
            "passed": bool(results["summary"].notna().all()),
        },
        {
            "check": "All categories valid",
            "passed": bool(results["category"].isin(NEWS_CATEGORIES).all()),
        },
        {
            "check": "No duplicate URLs",
            "passed": bool(not results["url"].dropna().duplicated().any()),
        },
    ]

    if "date_grounded" in results.columns:
        grounded = (
            results["date_grounded"]
            .astype(str)
            .str.lower()
            .map({"true": True, "false": False})
        )
        qa_rows.append({
            "check": "All new summaries pass date-grounding validation",
            "passed": bool(grounded.dropna().all()),
        })

    display(pd.DataFrame(qa_rows))

    # Review AAPL articles specifically for incidental relevance.
    aapl = results[
        results["ticker"].astype(str).str.upper() == "AAPL"
    ].copy()

    if not aapl.empty:
        review_cols = [
            c for c in [
                "published_at",
                "title",
                "relevance_score",
                "relevance_reason",
                "summary",
                "date_grounded",
            ]
            if c in aapl.columns
        ]
        display(aapl[review_cols])


,check,passed
0,All summaries present,True
1,All categories valid,True
2,No duplicate URLs,True
3,All new summaries pass date-grounding validation,True


,published_at,title,relevance_score,relevance_reason,summary,date_grounded
0,2026-09-20 16:37:00+00:00,"Prediction: Even With the $1,999 Price Tag, Apple's Foldable iPhone Duo Will Be a Hit and Apple Will Join Nvidia in the $5 Trillion Club.",2,NaN,"The main event is a prediction that Apple's foldable iPhone Duo, priced at $1,999, will be a hit and will see Apple join NVIDIA in the $...",NaN
1,2026-09-20 16:25:40+00:00,AAPL Looks 17.3% Overvalued on GF Value™ as Insider Selling Persists,2,NaN,Apple Inc. is reportedly 17.3% overvalued on GF Value™ as insider selling persists. The key facts are that AAPL is shown as 17.3% overva...,NaN
2,2026-09-20 11:30:29+00:00,Apple Enhances Durability with New A20 Pro Chip and Ceramic Shield in iPhone 18 Pro,2,NaN,"Apple unveiled the A20 Pro chip, Ceramic Shield in the iPhone 18 Pro, and an innovative folding screen on September 20, 2026. The key fa...",NaN
3,2026-09-19 12:50:00+00:00,John Ternus's First iPhone Launch Prompted a Bank of America Price Target Cut. Is Apple Stock a Buy?,2,NaN,"Bank of America cut Apple's price target after John Ternus's first iPhone launch event as Apple's CEO, where Apple unveiled the iPhone 1...",NaN
4,2026-09-18 18:11:54+00:00,Apple Raised Stakes with its $2000-Priced Foldable. But Can it Be the New Growth Engine?,2,NaN,"Apple released its first foldable phone, the Duo, priced at $2000. The Duo is Apple's first foldable phone and represents the most signi...",NaN
5,2026-09-18 17:57:20+00:00,AAPL Looks 16.9% Overvalued on GF Value™ Amid Strong Upgrade Cycle Outlook,2,NaN,Apple is viewed as 16.9% overvalued on GF Value™ amid a strong upgrade cycle outlook. GF Value™ indicates Apple (AAPL) is 16.9% overvalu...,NaN
6,2026-09-18 16:55:26+00:00,META Looks 21.3% Undervalued on GF Value™ as Muse App Tops Apple Store,2,NaN,"Meta Platforms Inc.'s Muse app topped Apple's U.S. App Store, reaching No. 1 on September 18, 2026. Key facts show Muse surged to No. 1 ...",NaN
7,2026-09-18 15:30:05+00:00,Apple Stock Has a New Opportunity Investors Can’t Afford to Ignore,2,NaN,"Apple reported its ninth consecutive earnings beat, including a ninth consecutive EPS beat. Shares are up 23% this year, and the company...",NaN
8,2026-09-18 13:30:58+00:00,AAPL Maintained by Evercore ISI Group -- Price Target Raised to $380,2,NaN,"Evercore ISI Group maintained an Outperform rating for Apple Inc (AAPL) and raised its price target to $380 on September 18, 2026. This ...",NaN
9,2026-09-18 13:30:00+00:00,"Apple Stock: The $1,999 iPhone Hides A Bigger Problem (NASDAQ:AAPL)",2,NaN,"An analyst argues that Apple stock is a sell due to a bigger problem hinted by the $1,999 iPhone. The article cites the $1,999 iPhone, n...",NaN


## 6. Planner, Router, and Specialist Agents

All agent functions below are included directly rather than imported from `src/`.

### 6.1 Planner Agent

In [16]:
"""Planner Agent for the investment research multi-agent system."""


def generate_research_plan(ticker: str) -> dict:
    """Create a structured research plan for a stock ticker."""

    ticker = ticker.upper().strip()

    research_steps = [
        {
            "step_id": 1,
            "task": "news_analysis",
            "description": f"Review recent financial news and macro developments relevant to {ticker}.",
        },
        {
            "step_id": 2,
            "task": "market_analysis",
            "description": f"Analyze recent price, return, and trading-volume behavior for {ticker}.",
        },
        {
            "step_id": 3,
            "task": "insider_analysis",
            "description": f"Review recent SEC Form 4 insider transactions for {ticker}.",
        },
        {
            "step_id": 4,
            "task": "risk_analysis",
            "description": f"Identify important risks using the available news, market, and insider evidence for {ticker}.",
        },
        {
            "step_id": 5,
            "task": "catalyst_analysis",
            "description": f"Identify possible catalysts or notable events affecting {ticker}.",
        },
        {
            "step_id": 6,
            "task": "final_synthesis",
            "description": f"Synthesize the research evidence into a concise investment-research summary for {ticker}.",
        },
    ]

    return {
        "ticker": ticker,
        "objective": f"Conduct structured multi-agent investment research for {ticker}.",
        "research_steps": research_steps,
    }


### 6.2 Router Agent

In [17]:
"""Router Agent for the investment research multi-agent system."""


def route_step(step: dict) -> dict:
    """Route a planner step to the appropriate specialist agent."""

    task = step.get("task", "").strip().lower()

    routing_map = {
        "news_analysis": "news_agent",
        "market_analysis": "market_agent",
        "insider_analysis": "insider_agent",
        "risk_analysis": "synthesis_agent",
        "catalyst_analysis": "synthesis_agent",
        "final_synthesis": "synthesis_agent",
    }

    selected_agent = routing_map.get(task, "synthesis_agent")

    reasons = {
        "news_agent": "This task depends primarily on processed financial-news evidence.",
        "market_agent": "This task depends primarily on market price, return, and volume evidence.",
        "insider_agent": "This task depends primarily on SEC Form 4 insider-transaction evidence.",
        "synthesis_agent": "This task requires combining evidence from multiple sources.",
    }

    return {
        "step_id": step.get("step_id"),
        "task": task,
        "agent": selected_agent,
        "reason": reasons[selected_agent],
        "description": step.get("description", ""),
    }


### 6.3 News Specialist

In [18]:
def analyze_news(ticker: str, news_df: pd.DataFrame) -> dict:
    """Analyze already-processed news research results for a ticker."""

    ticker = ticker.upper().strip()

    if news_df is None or news_df.empty:
        return {
            "agent": "news_agent",
            "ticker": ticker,
            "article_count": 0,
            "categories": {},
            "research_notes": [],
            "message": f"No processed news found for {ticker}.",
        }

    ticker_news = news_df[
        news_df["ticker"].astype(str).str.upper() == ticker
    ].copy()

    if ticker_news.empty:
        return {
            "agent": "news_agent",
            "ticker": ticker,
            "article_count": 0,
            "categories": {},
            "research_notes": [],
            "message": f"No processed news found for {ticker}.",
        }

    category_counts = ticker_news["category"].value_counts().to_dict()

    note_columns = [
        column
        for column in ["title", "category", "event", "summary", "url"]
        if column in ticker_news.columns
    ]

    research_notes = ticker_news[note_columns].to_dict(orient="records")

    return {
        "agent": "news_agent",
        "ticker": ticker,
        "article_count": len(ticker_news),
        "categories": category_counts,
        "research_notes": research_notes,
    }

### 6.4 Market Specialist

In [19]:
def analyze_market(ticker: str, market_df: pd.DataFrame) -> dict:
    """Analyze the already-cleaned market DataFrame for a ticker."""

    ticker = ticker.upper().strip()

    if market_df is None or market_df.empty:
        return {
            "agent": "market_agent",
            "ticker": ticker,
            "row_count": 0,
            "message": f"No market data found for {ticker}.",
        }

    ticker_market = market_df[
        market_df["ticker"].astype(str).str.upper() == ticker
    ].copy()

    if ticker_market.empty:
        return {
            "agent": "market_agent",
            "ticker": ticker,
            "row_count": 0,
            "message": f"No market data found for {ticker}.",
        }

    ticker_market["date"] = pd.to_datetime(
        ticker_market["date"],
        errors="coerce",
        utc=True,
    )
    ticker_market = ticker_market.sort_values("date")

    latest = ticker_market.iloc[-1]

    return {
        "agent": "market_agent",
        "ticker": ticker,
        "row_count": len(ticker_market),
        "start_date": ticker_market["date"].min(),
        "end_date": ticker_market["date"].max(),
        "latest_close": float(latest["close"]),
        "latest_volume": float(latest["volume"]),
        "average_daily_return": float(ticker_market["daily_return"].mean()),
        "average_volume": float(ticker_market["volume"].mean()),
    }

### 6.5 Insider Specialist

In [20]:
def analyze_insider_activity(ticker: str, insider_df: pd.DataFrame) -> dict:
    """Analyze the already-cleaned SEC Form 4 DataFrame for a ticker."""

    ticker = ticker.upper().strip()

    if insider_df is None or insider_df.empty:
        return {
            "agent": "insider_agent",
            "ticker": ticker,
            "transaction_count": 0,
            "message": f"No insider transactions found for {ticker}.",
        }

    ticker_insider = insider_df[
        insider_df["ticker"].astype(str).str.upper() == ticker
    ].copy()

    if ticker_insider.empty:
        return {
            "agent": "insider_agent",
            "ticker": ticker,
            "transaction_count": 0,
            "message": f"No insider transactions found for {ticker}.",
        }

    ticker_insider["transaction_date"] = pd.to_datetime(
        ticker_insider["transaction_date"],
        errors="coerce",
        utc=True,
    )
    ticker_insider["transaction_value"] = pd.to_numeric(
        ticker_insider["transaction_value"],
        errors="coerce",
    )
    ticker_insider["shares"] = pd.to_numeric(
        ticker_insider["shares"],
        errors="coerce",
    )

    acquired = ticker_insider[
        ticker_insider["acquired_disposed"] == "A"
    ]
    disposed = ticker_insider[
        ticker_insider["acquired_disposed"] == "D"
    ]

    return {
        "agent": "insider_agent",
        "ticker": ticker,
        "transaction_count": len(ticker_insider),
        "unique_insiders": ticker_insider["owner_name"].nunique(),
        "acquired_transactions": len(acquired),
        "disposed_transactions": len(disposed),
        "total_transaction_value": float(
            ticker_insider["transaction_value"].fillna(0).sum()
        ),
        "start_date": ticker_insider["transaction_date"].min(),
        "end_date": ticker_insider["transaction_date"].max(),
    }

### 6.6 Synthesis Agent

In [21]:
"""Synthesis specialist for combining multi-source research evidence."""


def synthesize_research(
    ticker: str,
    news_result: dict,
    market_result: dict,
    insider_result: dict,
) -> dict:
    """Combine specialist outputs into structured research conclusions."""

    ticker = ticker.upper().strip()

    risks = []
    catalysts = []

    # News-derived signals
    categories = news_result.get("categories", {})

    if categories.get("macro_market", 0) > 0:
        risks.append(
            "Macro or broader market developments may affect the stock."
        )

    if categories.get("analyst_investor", 0) > 0:
        catalysts.append(
            "Recent analyst or investor commentary may influence market expectations."
        )

    # Market-derived signals
    avg_return = market_result.get("average_daily_return")

    if avg_return is not None:
        if avg_return < 0:
            risks.append(
                "Average daily market return over the available period is negative."
            )
        else:
            catalysts.append(
                "Average daily market return over the available period is positive."
            )

    # Insider-derived signals
    acquired = insider_result.get("acquired_transactions", 0)
    disposed = insider_result.get("disposed_transactions", 0)

    if disposed > acquired:
        risks.append(
            "Disposed insider transactions outnumber acquired transactions in the available Form 4 data."
        )

    if acquired > 0:
        catalysts.append(
            "The available Form 4 data includes insider acquisition transactions."
        )

    return {
        "agent": "synthesis_agent",
        "ticker": ticker,
        "risks": risks,
        "catalysts": catalysts,
        "summary": {
            "news_articles": news_result.get("article_count", 0),
            "market_rows": market_result.get("row_count", 0),
            "insider_transactions": insider_result.get(
                "transaction_count", 0
            ),
        },
    }


### 6.7 Orchestrator

In [22]:
def orchestrate_research(
    ticker: str,
    news_df: pd.DataFrame,
    market_df: pd.DataFrame,
    insider_df: pd.DataFrame,
) -> dict:
    """Run planner, router, specialists, and synthesis using in-memory data."""

    ticker = ticker.upper().strip()

    plan = generate_research_plan(ticker)
    routing_decisions = [
        route_step(step)
        for step in plan["research_steps"]
    ]

    news_result = analyze_news(ticker, news_df)
    market_result = analyze_market(ticker, market_df)
    insider_result = analyze_insider_activity(ticker, insider_df)

    synthesis_result = synthesize_research(
        ticker,
        news_result,
        market_result,
        insider_result,
    )

    specialist_results = {
        "news_agent": news_result,
        "market_agent": market_result,
        "insider_agent": insider_result,
        "synthesis_agent": synthesis_result,
    }

    executed_steps = []

    for decision in routing_decisions:
        agent_name = decision["agent"]
        executed_steps.append(
            {
                "step_id": decision["step_id"],
                "task": decision["task"],
                "agent": agent_name,
                "reason": decision["reason"],
                "result": specialist_results[agent_name],
            }
        )

    return {
        "ticker": ticker,
        "plan": plan,
        "routing_decisions": routing_decisions,
        "executed_steps": executed_steps,
        "specialist_results": specialist_results,
        "final_synthesis": synthesis_result,
    }


def display_orchestration(result: dict) -> None:
    """Print a readable summary of planner/router execution."""

    print("=" * 72)
    print(f"Ticker: {result['ticker']}")
    print(f"Objective: {result['plan']['objective']}")
    print("=" * 72)

    for step in result["executed_steps"]:
        print(
            f"Step {step['step_id']}: "
            f"{step['task']} -> {step['agent']}"
        )

    print("=" * 72)
    print("Final synthesis")

    print("Risks:")
    for risk in result["final_synthesis"]["risks"]:
        print(f"- {risk}")

    print("Catalysts:")
    for catalyst in result["final_synthesis"]["catalysts"]:
        print(f"- {catalyst}")

### 6.8 Run Agent Workflow

In [23]:
available_tickers = sorted(
    set(processed["market"]["ticker"].dropna().astype(str).str.upper())
    | set(processed["news"]["ticker"].dropna().astype(str).str.upper())
    | set(processed["sec"]["ticker"].dropna().astype(str).str.upper())
)

if not available_tickers:
    raise ValueError("No tickers are available after preprocessing.")

TICKER = available_tickers[0]
print("Demo ticker:", TICKER)

orchestration = orchestrate_research(
    ticker=TICKER,
    news_df=results,
    market_df=processed["market"],
    insider_df=processed["sec"],
)

display_orchestration(orchestration)

routing_df = pd.DataFrame(orchestration["routing_decisions"])
display(
    routing_df[
        ["step_id", "task", "agent", "reason"]
    ]
)

Demo ticker: AAPL
Ticker: AAPL
Objective: Conduct structured multi-agent investment research for AAPL.
Step 1: news_analysis -> news_agent
Step 2: market_analysis -> market_agent
Step 3: insider_analysis -> insider_agent
Step 4: risk_analysis -> synthesis_agent
Step 5: catalyst_analysis -> synthesis_agent
Step 6: final_synthesis -> synthesis_agent
Final synthesis
Risks:
- Macro or broader market developments may affect the stock.
- Disposed insider transactions outnumber acquired transactions in the available Form 4 data.
Catalysts:
- Recent analyst or investor commentary may influence market expectations.
- Average daily market return over the available period is positive.
- The available Form 4 data includes insider acquisition transactions.


,step_id,task,agent,reason
0,1,news_analysis,news_agent,This task depends primarily on processed financial-news evidence.
1,2,market_analysis,market_agent,"This task depends primarily on market price, return, and volume evidence."
2,3,insider_analysis,insider_agent,This task depends primarily on SEC Form 4 insider-transaction evidence.
3,4,risk_analysis,synthesis_agent,This task requires combining evidence from multiple sources.
4,5,catalyst_analysis,synthesis_agent,This task requires combining evidence from multiple sources.
5,6,final_synthesis,synthesis_agent,This task requires combining evidence from multiple sources.


## 7. Evaluator-Optimizer

The evaluator scores factual grounding, completeness, and clarity. The optimizer then revises failed drafts using the evaluator feedback. Memory stores prior run outcomes by ticker.

### 7.1 Evaluator

In [24]:
evaluator_client = build_llm_client()
EVALUATOR_MODEL_NAME = LLM_MODEL

# Pydantic schemas for evaluation rubric and results
class EvaluationRubric(BaseModel):
    factual_grounding: int = Field(
        description="Score 1-10: Are stock prices, market returns, and SEC Form 4 transactions accurate and directly supported by evidence?"
    )
    completeness: int = Field(
        description="Score 1-10: Does the report cover both daily market metrics and processed news context?"
    )
    clarity_and_structure: int = Field(
        description="Score 1-10: Is the analysis well-organized, concise, and logically structured?"
    )

class EvaluationResult(BaseModel):
    overall_score: int = Field(
        description="Overall quality score from 1 to 10."
    )
    passed: bool = Field(
        description="True if overall_score >= 8 and factual_grounding >= 8, otherwise False."
    )
    failure_reason: Optional[str] = Field(
        default=None,
        description="Concise explanation of why the draft failed the quality pass gate (required if passed is False; set to None if passed is True)."
    )
    rubric: EvaluationRubric
    missing_elements: List[str] = Field(
        description="List of specific missing data points, SEC transaction details, or unaddressed news facts."
    )
    refinement_feedback: str = Field(
        description="Detailed critique and specific instructions for the Optimizer to improve the draft."
    )


# Evaluator prompts 
EVALUATOR_SYSTEM_PROMPT = """
You are an expert Financial Research Evaluation Agent.
Your job is to rigorously evaluate candidate stock research reports against raw evidence sources and output structured quality metrics.

Evaluation Criteria:
1. FACTUAL GROUNDING (0-10): Are all stock prices, SEC Form 4 trade values, dates, and claims strictly supported by the raw evidence? Deduct heavily for any invented figures.
2. COMPLETENESS (0-10): Does the report cover all available key data points in the raw evidence (market metrics, insider trades, news)? 
   - NOTE ON MISSING METRICS: If specific financial metrics (such as quarterly revenue) are NOT present in the raw evidence, do NOT penalize the draft for omitting them, provided the draft notes their absence or accurately summarizes all available evidence.
3. CLARITY & STRUCTURE (0-10): Is the report professional, well-formatted, and logical?

Pass Threshold:
- Overall Score must be >= 8/10.
- Factual Grounding must be >= 8/10.
"""

EVALUATOR_USER_PROMPT_TEMPLATE = """
--- CANDIDATE RESEARCH DRAFT ---
{current_draft}

--- RAW EVIDENCE SOURCE ---
{raw_evidence}

Please evaluate the draft against the raw evidence and return a structured EvaluationResult.
"""


# Evaluation function
def evaluate_draft(
    current_draft: str,
    raw_evidence: str,
    model: str = EVALUATOR_MODEL_NAME
) -> EvaluationResult:
    """
    Evaluates a candidate research draft against raw evidence using structured LLM output.
    
    Parameters:
        current_draft (str): The research report draft to evaluate.
        raw_evidence (str): Ground truth evidence (daily_evidence.csv / news_research_results.csv).
        model (str): The LLM model to execute evaluation.
        
    Returns:
        EvaluationResult: Structured result object containing scores, pass/fail status with reason, and critique.
    """
    user_message = EVALUATOR_USER_PROMPT_TEMPLATE.format(
        current_draft=current_draft,
        raw_evidence=raw_evidence
    )

    # Generate response from LLM
    response = evaluator_client.beta.chat.completions.parse(
        model=model,
        messages=[
            {"role": "system", "content": EVALUATOR_SYSTEM_PROMPT},
            {"role": "user", "content": user_message}
        ],
        response_format=EvaluationResult,
        # Set temperature to a low number for deterministic grading
        temperature=0.1
    )

    eval_result: EvaluationResult = response.choices[0].message.parsed
    return eval_result

### 7.2 Optimizer

In [25]:
optimizer_client = build_llm_client()
OPTIMIZER_MODEL_NAME = LLM_MODEL

OPTIMIZER_SYSTEM_PROMPT = """
You are an expert Financial Research Optimization Agent.
Your job is to refine an investment analysis report based on structured feedback from an Evaluator Agent.

CRITICAL RULES:
1. MANDATORY SECTIONING: You MUST include a dedicated section titled "## Financials & Revenue Performance". In this section, explicitly state all revenue, sales, and financial growth metrics found in the Raw Evidence Source.
2. ADDRESS EVALUATOR CRITIQUE: Carefully review 'Missing Elements to Add' and 'Main Failure Reason'. Locate these exact figures in the Raw Evidence and write them into the revised draft.
3. STRICT FACTUAL GROUNDING: Every figure, percentage, stock price, and SEC trade must be strictly grounded in the Raw Evidence. Do NOT fabricate numbers.
4. ABSENT DATA FALLBACK: If specific quarterly revenue dollar figures are not present in the Raw Evidence, explicitly write under the section header:
   "Quarterly Revenue: Exact quarterly revenue dollar figures were not recorded in the primary evidence dataset for this reporting period."

Formatting Guidelines:
1. NEVER output bracketed template placeholders like '[Insert Current Date]' or '[Insert Ticker]'.
2. Replace date placeholders with actual dates or omit them cleanly.
"""


OPTIMIZER_USER_PROMPT_TEMPLATE = """
You are refining a stock research report to pass a strict quality gate.

Current Draft:
{current_draft}

Evaluator Critique & Failure Reason:
- Main Failure Reason: {failure_reason}
- Missing Elements to Add: {missing_elements}
- Specific Refinement Feedback: {feedback}

Raw Evidence Source:
{raw_evidence}

Instructions:
1. Revise the current draft to directly integrate the missing elements listed above.
2. Ensure every added metric (prices, Form 4 trade values, news headlines) is strictly grounded in the Raw Evidence.
3. If a requested metric is not in the Raw Evidence, state its absence explicitly as instructed.
"""


def optimize_draft(
    current_draft: str,
    feedback: str,
    missing_elements: List[str],
    raw_evidence: str,
    overall_score: int = 0,
    memory_context: Optional[str] = None,
    failure_reason: str = None,
    model: str = OPTIMIZER_MODEL_NAME
) -> str:
    """
    Refines a candidate research draft using feedback and missing facts from the Evaluator.
    
    Parameters:
        current_draft (str): The candidate research draft to improve.
        feedback (str): Constructive critique instructions from EvaluationResult.
        missing_elements (List[str]): List of missing items identified by the Evaluator.
        raw_evidence (str): Ground truth data (from daily_evidence.csv / news_research_results.csv).
        overall_score (int): Previous quality score.
        memory_context (str): Historical run memory/notes for this ticker.
        model (str): LLM model identifier.
        
    Returns:
        str: The updated, higher-quality research report draft.
    """
    formatted_missing = "\n".join([f"- {item}" for item in missing_elements]) if missing_elements else "None noted."
    formatted_memory = memory_context if memory_context else "No historical notes for this ticker."
    formatted_failure_reason = failure_reason if failure_reason else "N/A"

    user_message = OPTIMIZER_USER_PROMPT_TEMPLATE.format(
        current_draft=current_draft,
        failure_reason=formatted_failure_reason,
        overall_score=overall_score,
        feedback=feedback,
        missing_elements=formatted_missing,
        raw_evidence=raw_evidence,
        memory_context=formatted_memory
    )

    response = optimizer_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": OPTIMIZER_SYSTEM_PROMPT},
            {"role": "user", "content": user_message}
        ],
        # Low temperature for accurate, grounded revisions
        temperature=0.3
    )

    refined_draft = response.choices[0].message.content.strip()
    return refined_draft

## 8. Persistent Memory

In [26]:
class ResearchMemoryStore:
    """
    Manages persistent, cross-run memory for stock research analyses.
    Stores historical evaluation scores, recurring critique notes, and key takeaways per ticker.
    """

    def __init__(self, filepath: str = "data/processed/memory_store.json"):
        self.filepath = filepath
        self._ensure_dir_exists()
        self.memory: Dict[str, List[Dict[str, Any]]] = self._load_memory()

    def _ensure_dir_exists(self) -> None:
        """Ensures the destination folder exists before reading or writing."""
        dir_name = os.path.dirname(self.filepath)
        if dir_name and not os.path.exists(dir_name):
            os.makedirs(dir_name, exist_ok=True)

    def _load_memory(self) -> Dict[str, List[Dict[str, Any]]]:
        """Loads memory data from JSON file if present."""
        if os.path.exists(self.filepath):
            try:
                with open(self.filepath, "r", encoding="utf-8") as f:
                    return json.load(f)
            except (json.JSONDecodeError, OSError) as e:
                print(f"[MemoryStore] Warning: Failed to load memory file ({e}). Starting fresh.")
                return {}
        return {}

    def _save_memory(self) -> None:
        """Persists memory data to JSON file."""
        try:
            with open(self.filepath, "w", encoding="utf-8") as f:
                json.dump(self.memory, f, indent=2, ensure_ascii=False)
        except OSError as e:
            print(f"[MemoryStore] Error saving memory to {self.filepath}: {e}")

    def get_ticker_memory(self, ticker: str, max_entries: int = 3) -> str:
        """
        Fetches formatted history for a given ticker to inject into prompts.
        
        Parameters:
            ticker (str): Stock symbol (e.g., 'AAPL', 'NVDA').
            max_entries (int): Number of most recent runs to include.
            
        Returns:
            str: Formatted memory text block for LLM context injection.
        """
        symbol = ticker.upper()
        ticker_history = self.memory.get(symbol, [])

        if not ticker_history:
            return f"No previous research memory recorded for {symbol}."

        # Take the N most recent entries
        recent_runs = ticker_history[-max_entries:]
        memory_lines = [f"=== HISTORICAL MEMORY FOR {symbol} ==="]

        for i, entry in enumerate(recent_runs, 1):
            timestamp = entry.get("timestamp", "Unknown Date")
            score = entry.get("score", "N/A")
            summary = entry.get("summary", "No summary provided.")
            lessons = entry.get("lessons_learned", "")

            line = f"Run #{i} [{timestamp}] - Final Quality Score: {score}/10\n  Summary: {summary}"
            if lessons:
                line += f"\n  Lessons Learned: {lessons}"
            memory_lines.append(line)

        return "\n".join(memory_lines)

    def save_run(
        self,
        ticker: str,
        timestamp: Optional[str],
        score: int,
        summary: str,
        lessons_learned: Optional[str] = None
    ) -> None:
        """
        Appends a completed research run record to the memory store.
        
        Parameters:
            ticker (str): Stock symbol.
            timestamp (str): Execution timestamp string.
            score (int): Final evaluation score (1-10).
            summary (str): Brief summary of run outcome and evaluator feedback.
            lessons_learned (str): Optional recurring critique or pitfall note to avoid next time.
        """
        symbol = ticker.upper()
        if symbol not in self.memory:
            self.memory[symbol] = []

        if not timestamp:
            timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        entry = {
            "timestamp": timestamp,
            "score": score,
            "summary": summary,
            "lessons_learned": lessons_learned or ""
        }

        self.memory[symbol].append(entry)
        self._save_memory()
        print(f"[MemoryStore] Saved run for {symbol} to {self.filepath}.")

    def clear_memory(self, ticker: Optional[str] = None) -> None:
        """Clears memory for a specific ticker or wipes the entire memory store."""
        if ticker:
            symbol = ticker.upper()
            if symbol in self.memory:
                del self.memory[symbol]
                self._save_memory()
                print(f"[MemoryStore] Cleared memory for {symbol}.")
        else:
            self.memory = {}
            self._save_memory()
            print("[MemoryStore] Cleared all memory records.")


### 8.1 Evaluation-Optimization Loop

In [27]:
# Run Evaluator-Optimizer workflow
def run_evaluator_optimizer_workflow(
    ticker: str,
    initial_draft: str,
    raw_evidence: str,
    max_iterations: int = 3,
    memory_store_path: str = "data/processed/memory_store.json"
) -> Tuple[str, EvaluationResult, List[Dict[str, Any]]]:
    """
    Orchestrates the Evaluator-Optimizer refinement loop and handles cross-run memory.
    
    Parameters:
        ticker (str): The target stock symbol (e.g., 'AAPL', 'NVDA').
        initial_draft (str): The raw draft produced by specialist research agents.
        raw_evidence (str): The combined daily evidence and news processing facts.
        max_iterations (int): Maximum number of evaluation-optimization passes (default: 3).
        memory_store_path (str): Path to persistent JSON memory file.
        
    Returns:
        Tuple containing:
            - final_draft (str): The final polished analysis report.
            - final_eval (EvaluationResult): The final evaluation rubric & score object.
            - history (List[Dict]): Detailed iteration history for auditing.
    """
    ticker = ticker.upper()
    memory_store = ResearchMemoryStore(filepath=memory_store_path)
    
    # Fetch historical run memory for ticker
    memory_context = memory_store.get_ticker_memory(ticker)
    print(f"[{ticker}] Loaded Memory Context:\n{memory_context}\n" + "-"*50)

    current_draft = initial_draft
    iteration_history: List[Dict[str, Any]] = []
    final_eval_result: Optional[EvaluationResult] = None

    # Evaluator-Optimizer Execution Loop
    for iteration in range(1, max_iterations + 1):
        print(f"[{ticker}] --- Starting Iteration {iteration}/{max_iterations} ---")
        
        # Evaluate current draft
        eval_result: EvaluationResult = evaluate_draft(
            current_draft=current_draft,
            raw_evidence=raw_evidence
        )
        final_eval_result = eval_result

        # Log iteration
        iteration_log = {
            "iteration": iteration,
            "draft": current_draft,
            "overall_score": eval_result.overall_score,
            "passed": eval_result.passed,
            "rubric": eval_result.rubric.model_dump(),
            "feedback": eval_result.refinement_feedback,
            "missing_elements": eval_result.missing_elements,
            "timestamp": datetime.datetime.now().isoformat()
        }
        iteration_history.append(iteration_log)

        print(f"[{ticker}] Iteration {iteration} Score: {eval_result.overall_score}/10 | Passed: {eval_result.passed} | Failure Reason: {eval_result.failure_reason}")

        # Check termination criteria
        if eval_result.passed or iteration == max_iterations:
            if eval_result.passed:
                print(f"[{ticker}] Quality threshold met on iteration {iteration}!")
            else:
                print(f"[{ticker}] Reached max iterations ({max_iterations}). Finalizing current draft.")
            break

        # Refine draft using optimizer
        print(f"[{ticker}] Refining draft based on evaluator feedback...")
        current_draft = optimize_draft(
            current_draft=current_draft,
            feedback=eval_result.refinement_feedback,
            missing_elements=eval_result.missing_elements,
            raw_evidence=raw_evidence,
            overall_score=eval_result.overall_score,
            memory_context=memory_context,
            failure_reason=eval_result.failure_reason,
        )

    # Update memory storage with final results
    run_summary = (
        f"Final Score: {final_eval_result.overall_score}/10 after {len(iteration_history)} iterations. "
        f"Evaluator Notes: {final_eval_result.refinement_feedback[:150]}..."
    )
    memory_store.save_run(
        ticker=ticker,
        timestamp=datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        score=final_eval_result.overall_score,
        summary=run_summary
    )
    print(f"[{ticker}] Persistent memory updated in {memory_store_path}.")

    return current_draft, final_eval_result, iteration_history

## 9. End-to-End Research Workflow

In [28]:
MAX_NEWS_NOTES = 5


def _format_date(value) -> str:
    """Return a date as YYYY-MM-DD, or 'n/a' if missing."""
    if value is None or pd.isna(value):
        return "n/a"
    return pd.Timestamp(value).strftime("%Y-%m-%d")


def _clean_text(value) -> str:
    """Return text with missing values replaced by an empty string."""
    if value is None or pd.isna(value):
        return ""
    return str(value).strip()


def build_research_draft(result: dict) -> str:
    """Create the initial report directly from specialist-agent outputs."""

    ticker = result["ticker"]
    news = result["specialist_results"]["news_agent"]
    market = result["specialist_results"]["market_agent"]
    insider = result["specialist_results"]["insider_agent"]
    synthesis = result["final_synthesis"]

    lines = [f"# Investment Research Draft: {ticker}", ""]

    lines.append("## Market Performance")
    if market.get("row_count", 0) > 0:
        lines.extend(
            [
                f"- Trading days analyzed: {market['row_count']} "
                f"({_format_date(market['start_date'])} to {_format_date(market['end_date'])})",
                f"- Latest close: ${market['latest_close']:,.2f} "
                f"on volume of {market['latest_volume']:,.0f} shares",
                f"- Average daily return: {market['average_daily_return']:.2%}",
                f"- Average daily volume: {market['average_volume']:,.0f} shares",
            ]
        )
    else:
        lines.append(f"- {market.get('message', 'No market data available.')}")
    lines.append("")

    lines.extend(
        [
            "## Financials & Revenue Performance",
            "Quarterly Revenue: Exact quarterly revenue dollar figures were not recorded "
            "in the primary evidence dataset for this reporting period.",
            "",
        ]
    )

    lines.append("## Insider Activity (SEC Form 4)")
    if insider.get("transaction_count", 0) > 0:
        lines.extend(
            [
                f"- Transactions: {insider['transaction_count']} by "
                f"{insider['unique_insiders']} insiders "
                f"({_format_date(insider['start_date'])} to {_format_date(insider['end_date'])})",
                f"- Acquisitions: {insider['acquired_transactions']}, "
                f"dispositions: {insider['disposed_transactions']}",
                f"- Total transaction value: ${insider['total_transaction_value']:,.2f}",
            ]
        )
    else:
        lines.append(f"- {insider.get('message', 'No insider transactions available.')}")
    lines.append("")

    lines.append("## News Coverage")
    if news.get("article_count", 0) > 0:
        categories = ", ".join(
            f"{name} ({count})"
            for name, count in news["categories"].items()
        )
        lines.append(f"- Articles analyzed: {news['article_count']}")
        lines.append(f"- Categories: {categories}")

        for note in news["research_notes"][:MAX_NEWS_NOTES]:
            title = _clean_text(note.get("title"))
            summary = (
                _clean_text(note.get("summary"))
                or _clean_text(note.get("event"))
            )
            category = _clean_text(note.get("category"))
            label = f"- {title} [{category}]"
            lines.append(f"{label}: {summary}" if summary else label)
    else:
        lines.append(f"- {news.get('message', 'No news available.')}")
    lines.append("")

    lines.append("## Risks")
    if synthesis["risks"]:
        lines.extend(f"- {risk}" for risk in synthesis["risks"])
    else:
        lines.append("- No risks identified from the available evidence.")
    lines.append("")

    lines.append("## Catalysts")
    if synthesis["catalysts"]:
        lines.extend(f"- {catalyst}" for catalyst in synthesis["catalysts"])
    else:
        lines.append("- No catalysts identified from the available evidence.")

    return "\n".join(lines)


def build_raw_evidence(
    ticker: str,
    daily_evidence_df: pd.DataFrame,
    news_results_df: pd.DataFrame,
) -> str:
    """Build evaluator evidence from the DataFrames already produced above."""

    ticker = ticker.upper().strip()

    t_daily = daily_evidence_df[
        daily_evidence_df["ticker"].astype(str).str.upper() == ticker
    ]

    if news_results_df is None or news_results_df.empty:
        t_news = pd.DataFrame()
    else:
        t_news = news_results_df[
            news_results_df["ticker"].astype(str).str.upper() == ticker
        ]

    daily_text = (
        t_daily.to_string(index=False)
        if not t_daily.empty
        else "No market/SEC evidence recorded."
    )
    news_text = (
        t_news.to_string(index=False)
        if not t_news.empty
        else "No news research evidence recorded."
    )

    return (
        f"\n=== DAILY MARKET & SEC FORM 4 EVIDENCE ({ticker}) ===\n"
        f"{daily_text}\n"
        f"\n=== PROCESSED NEWS RESEARCH EVIDENCE ({ticker}) ===\n"
        f"{news_text}\n"
    )


def run_full_research(
    orchestration_result: dict,
    daily_evidence_df: pd.DataFrame,
    news_results_df: pd.DataFrame,
    max_iterations: int = 3,
    memory_store_path: str | Path | None = None,
) -> dict:
    """Evaluate and refine the already-created orchestration result."""

    ticker = orchestration_result["ticker"]

    if memory_store_path is None:
        memory_store_path = (
            PATHS["processed"] / "memory_store.json"
        )

    initial_draft = build_research_draft(orchestration_result)
    raw_evidence = build_raw_evidence(
        ticker,
        daily_evidence_df,
        news_results_df,
    )

    final_report, final_evaluation, history = (
        run_evaluator_optimizer_workflow(
            ticker=ticker,
            initial_draft=initial_draft,
            raw_evidence=raw_evidence,
            max_iterations=max_iterations,
            memory_store_path=str(memory_store_path),
        )
    )

    return {
        "ticker": ticker,
        "orchestration": orchestration_result,
        "initial_draft": initial_draft,
        "raw_evidence": raw_evidence,
        "final_report": final_report,
        "final_evaluation": final_evaluation,
        "history": history,
    }

### 9.1 Run the Complete Workflow

In [29]:
if not LLM_API_KEY:
    raise ValueError(
        "LLM_API_KEY is required for the evaluator-optimizer portion."
    )

initial_draft = build_research_draft(orchestration)
raw_evidence = build_raw_evidence(
    TICKER,
    evidence,
    results,
)

print("INITIAL DRAFT")
print("=" * 80)
print(initial_draft)

MEMORY_PATH = PATHS["processed"] / "memory_store.json"

research = run_full_research(
    orchestration_result=orchestration,
    daily_evidence_df=evidence,
    news_results_df=results,
    max_iterations=3,
    memory_store_path=MEMORY_PATH,
)

history = pd.DataFrame(
    [
        {
            "iteration": step["iteration"],
            "overall_score": step["overall_score"],
            "passed": step["passed"],
            "factual_grounding": step["rubric"]["factual_grounding"],
            "completeness": step["rubric"]["completeness"],
            "clarity_and_structure": step["rubric"]["clarity_and_structure"],
        }
        for step in research["history"]
    ]
)

print("\nEVALUATION HISTORY")
display(history)

print("\nFINAL REPORT")
print("=" * 80)
print(research["final_report"])

memory_store = ResearchMemoryStore(
    filepath=str(MEMORY_PATH)
)

print("\nPERSISTENT MEMORY")
print("=" * 80)
print(memory_store.get_ticker_memory(TICKER))

INITIAL DRAFT
# Investment Research Draft: AAPL

## Market Performance
- Trading days analyzed: 100 (2026-04-28 to 2026-09-18)
- Latest close: $336.13 on volume of 86,588,203 shares
- Average daily return: 0.24%
- Average daily volume: 52,609,596 shares

## Financials & Revenue Performance
Quarterly Revenue: Exact quarterly revenue dollar figures were not recorded in the primary evidence dataset for this reporting period.

## Insider Activity (SEC Form 4)
- Transactions: 57 by 9 insiders (2026-02-24 to 2026-09-15)
- Acquisitions: 11, dispositions: 46
- Total transaction value: $168,657,491.72

## News Coverage
- Articles analyzed: 18
- Categories: analyst_investor (9), product_service (4), regulation_legal (3), earnings (1), macro_market (1)
- Prediction: Even With the $1,999 Price Tag, Apple's Foldable iPhone Duo Will Be a Hit and Apple Will Join Nvidia in the $5 Trillion Club. [product_service]: The main event is a prediction that Apple's foldable iPhone Duo, priced at $1,999, will b

,iteration,overall_score,passed,factual_grounding,completeness,clarity_and_structure
0,1,8,False,7,9,8
1,2,8,True,9,8,8



FINAL REPORT
Investment Research Draft: AAPL

Executive Summary (revised)
- The latest closing print on the dataset is 336.13 for AAPL on 2026-09-18 with 86,588,203 shares traded.
- Raw evidence confirms insider-transaction activity across 9 insiders totaling $168.657M, with 57 trades reported in the full period. A granular, date-by-date reconciliation is provided below.
- Revenue figures are not present in the primary evidence; no quarterly revenue dollars or growth metrics are available from the raw data.
- News coverage comprises 18 processed items spanning product launches, valuation signals, regulatory/legal matters, earnings framing, and macro-market context, with explicit source-date mappings in the appendix.

## Market Performance
- Latest close: 336.13 on 2026-09-18, volume: 86,588,203 shares.
- Grounding note: The dataset contains daily observations up to 2026-09-20; however, the current primary-evidence extract only confirms the single latest close and volume figures direct

## 10. Final Results and Discussion

This standalone notebook demonstrates all three required agentic workflow ideas within one end-to-end financial research system:

- **Prompt chaining:** financial news is classified, structured facts are extracted, and an evidence-grounded summary is generated.
- **Routing:** the Planner creates research tasks and the Router directs those tasks to news, market, insider, or synthesis specialists.
- **Evaluator–Optimizer:** a candidate research report is scored against raw evidence and revised until it passes the quality threshold or reaches the iteration limit.